### Part B question 4 - What is the change in word frequencies if normalization is done after stop word removal.

In [2]:
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [3]:
# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
df = pd.read_csv('Data (1).csv')

In [5]:
df.head()

,Data
0,Watch or listen live weekdays at 8:30am MT at ...
1,Watch or listen live weekdays at 8:30am MT at ...
2,"Chubby And Hot, Always Stir The Pot!"
3,"Chubby And Hot, Always Stir The Pot!"
4,"Journalist, publisher of Rebel News — telling ..."


In [6]:
# Initialize tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [7]:
def tokenize(text):
    return [word for word in word_tokenize(text.lower()) if word.isalpha()]


### We will first remove stopwords then normalize

In [9]:
#Stopword Removal → Lemmatize
def Stop_normalize(text):
    tokens = tokenize(text)
    no_stopwords = [w for w in tokens if w not in stop_words]
    return [lemmatizer.lemmatize(w) for w in no_stopwords]


In [10]:
df['Stop_normalize'] = df['Data'].apply(Stop_normalize)

## Now we will first normalize and then remove stopwords

In [12]:
# Lemmatize → Stopword Removal
def normalize_stop(text):
    tokens = tokenize(text)
    lemmatized = [lemmatizer.lemmatize(w) for w in tokens]
    return [w for w in lemmatized if w not in stop_words]


In [13]:
df['normalize_stop'] = df['Data'].apply(normalize_stop)

In [14]:
# Show side-by-side difference
pd.set_option('display.max_colwidth', None)
df[['Data', 'Stop_normalize', 'normalize_stop']].sample(5)

,Data,Stop_normalize,normalize_stop
3933,Oilers fan Twins fan Vikings fan. father of 2 girls.,"[oiler, fan, twin, fan, viking, fan, father, girl]","[oiler, fan, twin, fan, viking, fan, father, girl]"
5198,"Outdoor enthusiast, community volunteer, lover of good wine and craft beer.","[outdoor, enthusiast, community, volunteer, lover, good, wine, craft, beer]","[outdoor, enthusiast, community, volunteer, lover, good, wine, craft, beer]"
7108,Your City. Now. \nYour home base for everything Edmonton. Send stories/leads to edmonton@dailyhive.com,"[city, home, base, everything, edmonton, send, edmonton]","[city, home, base, everything, edmonton, send, edmonton]"
1344,Former journalist and now publisher of tourism magazines.,"[former, journalist, publisher, tourism, magazine]","[former, journalist, publisher, tourism, magazine]"
4812,CORE (Calgary Outdoor Recreation Enthusiasts Society) is a Calgary-based hiking club created in 1999. A small club of very active members of varying ages.,"[core, calgary, outdoor, recreation, enthusiast, society, hiking, club, created, small, club, active, member, varying, age]","[core, calgary, outdoor, recreation, enthusiast, society, hiking, club, created, small, club, active, member, varying, age]"


In [15]:
from collections import Counter
import pandas as pd

# Flatten the lists into a single list of words
words_stop_normalize = [word for sublist in df['Stop_normalize'] for word in sublist]
words_normalize_stop = [word for sublist in df['normalize_stop'] for word in sublist]

# Count word frequencies
freq_stop_normalize = Counter(words_stop_normalize)
freq_normalize_stop = Counter(words_normalize_stop)

# Convert to DataFrames
df_freq_stop_normalize = pd.DataFrame(freq_stop_normalize.items(), columns=['Word', 'Freq_Stop_Normalize'])
df_freq_normalize_stop = pd.DataFrame(freq_normalize_stop.items(), columns=['Word', 'Freq_Normalize_Stop'])

# Merge on Word
df_comparison = pd.merge(df_freq_stop_normalize, df_freq_normalize_stop, on='Word', how='outer').fillna(0)

# Sort by max frequency difference
df_comparison['Difference'] = abs(df_comparison['Freq_Stop_Normalize'] - df_comparison['Freq_Normalize_Stop'])
df_comparison = df_comparison.sort_values(by='Difference', ascending=False)


##  Effect of Preprocessing Order: Word Frequency Comparison

This table compares the frequency of words extracted using two different text preprocessing pipelines:

- **`Stop_Normalize`**: Stopword removal followed by lemmatization  
- **`Normalize_Stop`**: Lemmatization followed by stopword removal

In [17]:
# Show top 20 words with most frequency difference
print(df_comparison.head(20))

             Word  Freq_Stop_Normalize  Freq_Normalize_Stop  Difference
2577           ha                  6.0                 81.0        75.0
6056           wa                  0.0                 48.0        48.0
1668          doe                  0.0                 19.0        19.0
348            as                 12.0                  0.0        12.0
2856           in                  6.0                  0.0         6.0
247            an                  2.0                  0.0         2.0
0             aaa                  2.0                  2.0         0.0
4311     politics                142.0                142.0         0.0
4319       poolie                  1.0                  1.0         0.0
4318       poodle                  6.0                  6.0         0.0
4317       ponoka                 12.0                 12.0         0.0
4316   ponderings                  4.0                  4.0         0.0
4315      pompous                  2.0                  2.0     

##  Key Insights

### 1. Unexpected Tokens in `Normalize_Stop`
- Words like `'ha'`, `'wa'`, and `'doe'` appear with **high frequency** in `Normalize_Stop` but are low or absent in `Stop_Normalize`.
- These are likely **lemmatized forms of stopwords** (e.g., `has`, `was`, `does`) that:
  - Were transformed into non-informative tokens by the lemmatizer.
  - Were not recognized as stopwords after lemmatization and remained.



### 2. Missing Stopwords in `Normalize_Stop`
- Tokens like `'as'`, `'in'`, and `'an'` are removed in `Normalize_Stop` but appear in `Stop_Normalize`.
- These words are not altered by lemmatization, so if stopwords are removed **after**, they are most likely suffureing from formating issues like (like extra whitespace, casing, or list mismatch)..


### 3. Consistent Words Across Both Pipelines
- Words like `'poodle'`, `'ponoka'`, and `'politics'` have **identical frequencies**, showing they are unaffected by processing order.
- These represent clean, informative tokens that are well-preserved.


### ChatGPT Queries used

#### 1st query) normalization after stopwords or stopwords adter normalization , which is better and how it impacts word frequency.
#### last query )why words like as , in and an are appering even after removing stop word , when we do normalization and then stopword removal ?
